In [0]:
!pip install -qU langchain-openai

In [0]:
dbutils.library.restartPython()

In [0]:
from typing import List
from pydantic import BaseModel, Field

In [0]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str = Field(description="airport code to get weather for")

In [0]:
from langchain_core.utils.function_calling import convert_to_openai_function

In [0]:
weather_function = convert_to_openai_function(WeatherSearch)
weather_function

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_openai import ChatOpenAI

In [0]:
base_url = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}/serving-endpoints'
databricks_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [0]:
model = ChatDatabricks(
    endpoint = 'my-gpt-endpoint',
    extra_params = {"temperature": 0.1}
)
base_url = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}/serving-endpoints'
databricks_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

model = ChatOpenAI(
    model="my-gpt-endpoint",
    temperature=0,
    # max_tokens=None,
    # timeout=None,
    # max_retries=2,
    api_key=databricks_token,  # if you prefer to pass api key in directly instaed of using env vars
    base_url=base_url,
    # organization="...",
    # other params...
)

In [0]:
model.invoke("What's the weather in SF today?", functions=[weather_function])

In [0]:
model_with_function = model.bind(functions=[weather_function])
model_with_function.invoke("What's the weather in Bangalore today?")

* Forcing it to use a function

In [0]:
model_with_forced_function = model.bind(functions=[weather_function], function_call={"name":"WeatherSearch"})

In [0]:
model_with_forced_function.invoke("what is the weather in sf?")

In [0]:
model_with_forced_function.invoke("hi!")

* Using a Chain

In [0]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "{input}")
])

In [0]:
chain = prompt | model_with_function

In [0]:
chain.invoke({"input": "what is the weather in sf?"})

### Using Multiple Functions

In [0]:
class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""
    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results")

In [0]:
functions = [
  convert_to_openai_function(WeatherSearch),
  convert_to_openai_function(ArtistSearch)
]

In [0]:
model_with_functions = model.bind(functions=functions)

In [0]:
model_with_functions.invoke("what is the weather in sf?")

In [0]:
model_with_functions.invoke("what are three songs by Akon?")

In [0]:
model_with_functions.invoke("hi!")